**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Blind Source Separation & ICA

The cocktail-party problem, actually solved: several microphones each hear a *mixture* of sources, and — knowing nothing about the mixing — we unmix them. The key is a beautiful statistical loophole: Gaussianity is the one thing mixing *increases*. Verified the only way that matters: recovered sources correlate ≈1 with the planted truth.

## 1. Pre-requisites

[Statistical SP](./Statistical_Signal_Processing.ipynb), [Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) S3, [Independence](../Intro_Math/Analysis/Independence.ipynb).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as sig
rng = np.random.default_rng(0)

# three planted sources: a chirp 'voice', a square-wave 'hum', an impulsive 'percussion'
fs, T_dur = 8000, 3.0
t = np.arange(0, T_dur, 1/fs)
s1 = sig.chirp(t, 300, T_dur, 800) * (1 + 0.3*np.sin(2*np.pi*2*t))
s2 = sig.square(2*np.pi*120*t) * 0.7
s3 = np.zeros_like(t)
for tc in rng.uniform(0, T_dur, 25):
    i = int(tc*fs); s3[i:i+150] += np.exp(-np.arange(150)/25) * rng.choice([-2, 2])
S_true = np.stack([s1, s2, s3])
S_true = (S_true - S_true.mean(1, keepdims=True)) / S_true.std(1, keepdims=True)

A_mix = rng.standard_normal((3, 3))                    # unknown room acoustics
X = A_mix @ S_true                                      # what the microphones record

---
### 🕐 Session 1 of 3 — *The Problem & Why Correlation Isn't Enough* (~35 min)
**Goal:** see mixing destroy the sources; understand why PCA/whitening only gets you halfway.
**Feeds into:** Session 2 (the non-Gaussian loophole).

---

## 2. Three Microphones, Three Tangles

💡 **Intuition.** Each mic hears $x_i = \sum_j a_{ij} s_j$: linear, instantaneous mixing. **Whitening** (decorrelating via the covariance [eigendecomposition](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb)) can undo mixing *up to a rotation* — but second-order statistics are **rotation-blind**: every rotation of white signals is equally white. Correlation has taken you to a sphere of candidate unmixings and gone silent. Something beyond variance must pick the rotation — that something is Session 2.

In [ ]:
# whiten

# YOUR CODE HERE


---
### 🕐 Session 2 of 3 — *The Non-Gaussian Loophole & FastICA* (~40 min)
**Goal:** the CLT in reverse: mixtures are MORE Gaussian than sources — so maximize non-Gaussianity.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (limits & practice).

---

## 3. Gaussianity as a Compass

💡 **Intuition.** The [CLT](../Intro_Math/Analysis/Independence.ipynb) says sums of independent things drift *toward* Gaussian. Flip it around: each microphone (a sum of sources) is **more Gaussian than any single source** — so to unmix, rotate the whitened data until each output is as **non-Gaussian as possible**. That's all of ICA. FastICA does it with a fixed-point iteration on a smooth non-Gaussianity score (we use $\log\cosh$), one source at a time, deflating (Gram–Schmidt) so each new direction is orthogonal to the found ones. Built-in limits fall out of the logic: source order and sign/scale are unrecoverable, and **two Gaussian sources cannot be separated** (their mixtures are exactly rotation-symmetric).

In [ ]:
# ORACLE: each recovered source must match ONE true source with |corr| ≈ 1

# YOUR CODE HERE


In [ ]:

# YOUR CODE HERE


---
### 🕐 Session 3 of 3 — *Limits, Diagnostics & Practice* (~30 min)
**Goal:** what ICA can't do, how to sanity-check it, and where it runs in the wild.
**Builds on:** Session 2.

---

## 4. The Honest Fine Print

In [ ]:
# the promised failure: two GAUSSIAN sources are unseparable — watch it happen

# YOUR CODE HERE


**Field guide.**

- **Works:** EEG artifact removal (eye blinks are gloriously non-Gaussian), [audio](./Audio_Speech_DSP.ipynb) unmixing with instantaneous mixtures, hyperspectral unmixing.
- **Fails or needs upgrades:** convolutive/reverberant mixing (rooms delay, not just scale — needs frequency-domain ICA), more sources than mics (underdetermined → [sparsity](./Sparse_Dictionary_Learning.ipynb) to the rescue), Gaussian-ish sources.
- **Diagnostics:** always check kurtosis of outputs (should be far from 0), and run restarts — consistent answers across restarts are the practical identifiability certificate.
- **Lineage:** [contrastive learning](../Intro_Mach_Learn/Representation_Learning.ipynb) and modern disentanglement research are ICA's descendants (nonlinear ICA is provably impossible without auxiliary structure — a live research frontier).

## 5. Conclusion

Whitening gets you to a rotation; the CLT-in-reverse picks it; FastICA computes it (recovered × truth correlations > 0.99, verified); and Gaussian sources mark the hard boundary of the possible (also verified). Blindness, it turns out, is negotiable — Gaussianity isn't.

---
## Where next

- [Array Processing](./Array_Processing.ipynb) — unmixing with *geometry* instead of statistics.
- [Representation Learning](../Intro_Mach_Learn/Representation_Learning.ipynb) — the neural descendants.
- [Manifold Optimization](../Intro_Math/Optimization/Manifold_Optimization.ipynb) — ICA's rotation search lives on the Stiefel manifold.